In [1]:
from pathlib import Path
import warnings

import scanpy as sc
import scib
import numpy as np
import sys
import anndata as ad 
import scgpt as scg
import matplotlib.pyplot as plt

plt.style.context('default')
warnings.simplefilter("ignore", ResourceWarning)

model_dir = Path("../../scGPT_CP")

/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [2]:
# Set Base directory to the location of this script
BASE = Path('/mnt/hd1/home/ankitray/scFM/')

# Set working directory to the location of this script 
# data_dir = BASE / 'h5s_common_directory'
liver_data_dir = BASE / 'data/liver_cell_type_adata'

In [9]:
# Read in each h5ad file with the Anndata object variable name being it's file name without the extension and store in dictionary
adata_dict = {}

for h5ad_file in liver_data_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary

/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.

In [10]:
gene_col = "Gene Symbol"
cell_type_key = "typist_liver_majority_voting"
batch_key = "GSE"
N_HVG = 5000

In [17]:
for adata_name, adata in adata_dict.items():
    print(adata)

AnnData object with n_obs × n_vars = 39486 × 5000
    obs: 'barcode', 'gsm_id', 'sample_name', 'sample_title', 'condition', 'donor', 'batch', 'Sex', 'Age', 'New_Sample_ID', 'Patient_ID', 'assay_type', 'n_genes', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'typist_liver_predicted_labels', 'typist_liver_over_clustering', 'typist_liver_majority_voting', 'typist_liver_conf_score', 'sample_id', 'species', 'assay', 'dataset', 'sex', 'age', 'ethnicity', 'Cause_of_death', 'alcohol_use', 'sample', 'Sample name', 'title', 'source name', 'organism', 'characteristics: shortFileName', 'characteristics: strain', 'characteristics: platform', 'characteristics: digestion method', 'characteristics: number of added ABs', 'characteristic

In [16]:
for adata_name, adata in adata_dict.items():
    adata_dict[adata_name].var[gene_col] = adata.var.index.values

/tmp/ipykernel_680134/4091491784.py:2: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata_dict[adata_name].var[gene_col] = adata.var.index.values
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_680134/4091491784.py:2: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata_dict[adata_name].var[gene_col] = adata.var.index.values
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not un

In [12]:
# For each adata in adata_dict find the value counts per GSE and print the value counts
for adata_name, adata in adata_dict.items():
    print(f"Value counts for {adata_name}:")
    print(adata.obs[batch_key].value_counts())

Value counts for Endothelial cells:
GSE
GSE212837          20344
GSE202379           8876
GSE192740_liver     3361
GSE185477           3087
GSE189600           2400
GSE174748           1418
Name: count, dtype: int64
Value counts for Macrophages:
GSE
GSE212837          5666
GSE202379          1410
GSE192740_liver    1082
GSE174748           479
GSE185477           461
GSE189600            24
Name: count, dtype: int64
Value counts for T cells:
GSE
GSE202379          2055
GSE185477          1893
GSE212837          1460
GSE174748           370
GSE192740_liver     207
Name: count, dtype: int64
Value counts for Resident NK:
GSE
GSE212837          344
GSE202379          230
GSE174748           98
GSE192740_liver     63
Name: count, dtype: int64
Value counts for Cholangiocytes:
GSE
GSE212837          5079
GSE189600          3394
GSE202379          2745
GSE192740_liver     790
GSE185477           457
GSE174748           451
Name: count, dtype: int64
Value counts for Hepatocytes:
GSE
GSE212837  

In [13]:
for adata_name, adata in adata_dict.items():
    print(f"Processing {adata_name}...")
    # Could be flavor seurat_v3 with raw counts layer or cell_ranger
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat_v3", layer="raw_counts")
    adata = adata[:, adata.var['highly_variable']]
    print(adata.shape)
    adata_dict[adata_name] = adata  # Update the dictionary with the filtered Anndata object

Processing Endothelial cells...
(39486, 5000)
Processing Macrophages...
(9122, 5000)
Processing T cells...
(5985, 5000)
Processing Resident NK...
(735, 5000)
Processing Cholangiocytes...
(12916, 5000)
Processing Hepatocytes...
(345203, 5000)
Processing Fibroblasts...
(16599, 5000)


In [14]:
import torch
# Check if Torch Cuda is available
torch.cuda.is_available()

True

In [ ]:
model_dir = Path("../../scGPT_CP")
for adata_name, adata in adata_dict.items():
    print(f"Generating embeddings for {adata_name}...")
    embed_adata = scg.tasks.embed_data(
        adata,
        model_dir,
        gene_col=gene_col,
        batch_size=64,
    )
    adata_dict[adata_name] = embed_adata  # Update the dictionary with the embedded Anndata object




Generating embeddings for Endothelial cells...
scGPT - INFO - match 4894/5000 genes in vocabulary of size 60697.


/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 617/617 [03:22<00:00,  3.05it/s]
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Generating embeddings for Macrophages...
scGPT - INFO - match 4895/5000 genes in vocabulary of size 60697.


/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 143/143 [01:07<00:00,  2.11it/s]
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Generating embeddings for T cells...
scGPT - INFO - match 4919/5000 genes in vocabulary of size 60697.


/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 94/94 [00:37<00:00,  2.49it/s]
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.

Generating embeddings for Resident NK...
scGPT - INFO - match 4913/5000 genes in vocabulary of size 60697.


/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells: 100%|██████████| 12/12 [00:23<00:00,  1.94s/it]
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings
/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Generating embeddings for Cholangiocytes...
scGPT - INFO - match 4909/5000 genes in vocabulary of size 60697.


/home/ankitray/.conda/envs/scgpt/lib/python3.10/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(
Embedding cells:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
# Set Base directory to the grandparent location of this script
BASE = Path('/mnt/hd1/home/ankitray/scFM/')

out_path = BASE / "data/liver_cell_type_adata_embedded/"
# Make directory if it doesn't exist
out_path.mkdir(parents=True, exist_ok=True)

for adata_name, adata in adata_dict.items():
    print(f"Saving {adata_name}...")
    adata.write_h5ad(out_path / f"{adata_name}_embedded.h5ad")
